# 🚀 CareBridge AI - Kaggle GPU Deployment (IMPROVED)

## What This Does:
1. ✅ Checks GPU availability
2. ✅ Clones latest code from GitHub
3. ✅ Installs all dependencies
4. ✅ Downloads MedGemma model (cached for future runs)
5. ✅ Starts FastAPI server
6. ✅ Exposes publicly via ngrok

---

## 🔧 Before Running:

**Kaggle Settings:**
- ⚙️ Accelerator: **GPU T4 x2**
- 🌐 Internet: **ON**
- 💾 Persistence: **Files only**

**Ngrok Setup:**
1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken
2. Sign up (free)
3. Copy your auth token
4. Paste in **Cell 11** (replace `YOUR_NGROK_TOKEN`)

---

## ⏱️ Timing:
- **First run:** ~15-20 minutes (model download)
- **Subsequent runs:** ~3-5 minutes (from cache)

---

## 📝 Cell Order:
**Run cells in order (1 → 12)**

Don't skip cells!

---
## ✅ CELL 1: Check GPU

In [1]:
import subprocess

print("🔍 Checking GPU availability...\n")

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print('\n❌ ERROR: GPU not found!')
    print('   → Go to: Notebook Settings → Accelerator → Select GPU T4 x2')
    print('   → Then restart kernel\n')
elif 'T4' in result.stdout:
    print('\n✅ Tesla T4 GPU detected - Ready to go!\n')
elif 'A100' in result.stdout:
    print('\n✅ A100 GPU detected - Even better!\n')
else:
    print('\n⚠️  GPU found but not T4/A100')
    print('   → Recommended: Switch to T4 for consistency\n')


🔍 Checking GPU availability...

Mon Apr  6 18:05:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------

---
## 📦 CELL 2: Install System Packages

In [2]:
%%bash
echo "📦 Installing system packages..."
apt-get update -qq
apt-get install -y -qq tesseract-ocr poppler-utils libgl1-mesa-glx > /dev/null 2>&1
echo "✅ System packages installed!"


📦 Installing system packages...
✅ System packages installed!


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


---
## 📂 CELL 3: Clone GitHub Repository

In [4]:
import os
import subprocess

REPO = "https://github.com/AVP2011/CareBridge_AI"
DEST = "/kaggle/working/carebridge"

print("📂 Setting up repository...\n")

if os.path.exists(f"{DEST}/.git"):
    print("   Repository exists - pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", DEST, "pull"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
else:
    print("   Cloning repository...")
    result = subprocess.run(
        ["git", "clone", REPO, DEST],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("   ✅ Clone successful!")
    else:
        print(f"   ❌ Clone failed: {result.stderr}")

print("\n📁 Repository contents:")
subprocess.run(["ls", "-la", DEST])

print("\n✅ Repository ready!")


📂 Setting up repository...

   Cloning repository...
   ✅ Clone successful!

📁 Repository contents:
total 72
drwxr-xr-x 12 root root 4096 Apr  6 18:07 .
drwxr-xr-x  4 root root 4096 Apr  6 18:07 ..
drwxr-xr-x  4 root root 4096 Apr  6 18:07 carebridge-ui
drwxr-xr-x  2 root root 4096 Apr  6 18:07 config
-rw-r--r--  1 root root  755 Apr  6 18:07 diagnostic_llm.py
drwxr-xr-x  2 root root 4096 Apr  6 18:07 engines
drwxr-xr-x  8 root root 4096 Apr  6 18:07 .git
-rw-r--r--  1 root root  303 Apr  6 18:07 .gitignore
drwxr-xr-x  2 root root 4096 Apr  6 18:07 llm
-rw-r--r--  1 root root 6996 Apr  6 18:07 main.py
drwxr-xr-x  2 root root 4096 Apr  6 18:07 ocr
drwxr-xr-x  3 root root 4096 Apr  6 18:07 rag
-rw-r--r--  1 root root 1620 Apr  6 18:07 README.md
-rw-r--r--  1 root root  974 Apr  6 18:07 requirements.txt
drwxr-xr-x  2 root root 4096 Apr  6 18:07 safety
drwxr-xr-x  2 root root 4096 Apr  6 18:07 schemas
drwxr-xr-x  2 root root 4096 Apr  6 18:07 services

✅ Repository ready!


---
## 🐍 CELL 4: Install Python Dependencies

In [5]:
import subprocess
import sys
!pip install pyngrok
 
BACKEND = "/kaggle/working/carebridge"

print("🐍 Installing Python packages...\n")
print("   This may take 2-3 minutes...\n")

# Install from requirements.txt
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", f"{BACKEND}/requirements.txt", "-q"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ All dependencies installed!\n")
else:
    print(f"⚠️  Some warnings (usually safe to ignore):\n{result.stderr[:500]}\n")

# Verify key packages
print("📋 Verifying key packages:")
packages = ["fastapi", "uvicorn", "transformers", "torch", "pyngrok"]
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", pkg],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        version = [line for line in result.stdout.split('\n') if 'Version:' in line][0]
        print(f"   ✅ {pkg}: {version.split(':')[1].strip()}")
    else:
        print(f"   ❌ {pkg}: NOT FOUND")

print("\n✅ Dependencies ready!")


🐍 Installing Python packages...

   This may take 2-3 minutes...

⚠️  Some warnings (usually safe to ignore):
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python

📋 Verifying key packages:
   ✅ fastapi: 0.123.10
   ✅ uvicorn: 0.40.0
   ✅ transformers: 5.2.0
   ✅ torch: 2.9.0+cu126
   ✅ pyngrok: 8.0.0

✅ Dependencies ready!


---
## 🤖 CELL 5: Check Model Cache

In [6]:
import os

CACHE_DIR = "/kaggle/working/hf_cache"

print("🔍 Checking HuggingFace cache...\n")

if os.path.exists(f"{CACHE_DIR}/hub"):
    print("✅ Model cache exists!")
    print(f"   Location: {CACHE_DIR}/hub")
    
    # Check cache size
    result = subprocess.run(
        ["du", "-sh", f"{CACHE_DIR}/hub"],
        capture_output=True,
        text=True
    )
    print(f"   Size: {result.stdout.split()[0]}")
    print("   → Model will load from cache (faster)\n")
else:
    print("⚠️  No cache found")
    print("   → Model will be downloaded (~4-5 GB)")
    print("   → This will take 10-15 minutes")
    print("   → Future runs will be much faster!\n")

# Create cache directory if doesn't exist
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"✅ Cache directory ready: {CACHE_DIR}")


🔍 Checking HuggingFace cache...

⚠️  No cache found
   → Model will be downloaded (~4-5 GB)
   → This will take 10-15 minutes
   → Future runs will be much faster!

✅ Cache directory ready: /kaggle/working/hf_cache


---
## 🧠 CELL 6: Pre-download Model (Optional but Recommended)

**This downloads the model NOW so server startup is faster.**

Skip this if you want the model to download during server start instead.

In [7]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/kaggle/working/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache/hub"

print("\U0001f9e0 Pre-downloading MedGemma model...\n")
print("   Model: google/medgemma-4b-it")
print("   Size: ~4-5 GB")
print("   Time: 10-15 minutes (first time only)\n")

# ── HuggingFace Authentication ──
from huggingface_hub import login

hf_token = None

# Method 1: Try Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("\u2705 Got HF_TOKEN from Kaggle Secrets!")
except Exception as e1:
    print(f"   Kaggle Secrets lookup failed: {e1}")

# Method 2: Try environment variable
if not hf_token:
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HF_TOKENS")
    if hf_token:
        print("\u2705 Got HF_TOKEN from environment variable!")

# Method 3: Hardcoded fallback (user can paste their token here)
if not hf_token:
    # \u26a0\ufe0f PASTE YOUR TOKEN BELOW if secrets don't work:
    # hf_token = "hf_YourTokenHere"
    pass

if hf_token:
    login(hf_token)
    print("\u2705 Logged in to HuggingFace!\n")
else:
    print("\u274c HF_TOKEN NOT FOUND!")
    print("   Fix: Go to Kaggle sidebar -> Add-ons -> Secrets")
    print("   Add Label: HF_TOKEN  Value: your huggingface token")
    print("   Then toggle it ON for this notebook and re-run.\n")
    raise RuntimeError("Cannot proceed without HF_TOKEN")

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "google/medgemma-4b-it"

try:
    print("\U0001f4e5 Downloading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("   \u2705 Tokenizer downloaded\n")

    print("\U0001f4e5 Downloading model (this is the slow part)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    print("   \u2705 Model downloaded and cached!\n")

    # Test inference
    print("\U0001f9ea Testing model inference...")
    test_input = "What is insurance waiting period?"
    inputs = tokenizer(test_input, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"   Test query: {test_input}")
    print(f"   Response preview: {response[:100]}...")
    print("\n\u2705 Model is working correctly!\n")

    # Clean up to free memory
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print("\U0001f9f9 Memory cleared for server startup\n")

except Exception as e:
    print(f"\n\u274c Error: {str(e)}\n")
    print("   If this fails, the model will download when server starts.\n")

print("\u2705 Model pre-download complete!")


🧠 Pre-downloading MedGemma model...

   Model: google/gemma-2-2b-it
   Size: ~4-5 GB
   Time: 10-15 minutes (first time only)



📥 Downloading tokenizer...

❌ Error: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
401 Client Error. (Request ID: Root=1-69d3f7ad-6146882e7d686432025180e3;19427bd9-020b-413d-82da-e33310392eb2)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

   If this fails, the model will download when server starts.

✅ Model pre-download complete!


---
## 🛠️ CELL 7: Check Server Configuration

In [26]:
BACKEND = "/kaggle/working/carebridge"

print("🛠️  Checking server configuration...\n")

# Check if main.py exists
if os.path.exists(f"{BACKEND}/main.py"):
    print("✅ main.py found")
    
    # Show first 30 lines
    print("\n📄 First 30 lines of main.py:\n")
    with open(f"{BACKEND}/main.py", 'r') as f:
        lines = f.readlines()[:30]
        for i, line in enumerate(lines, 1):
            print(f"{i:3d}: {line}", end='')
else:
    print("❌ main.py NOT FOUND!")
    print("   → Check if repo cloned correctly in Cell 3\n")

print("\n✅ Configuration check complete!")


🛠️  Checking server configuration...

✅ main.py found

📄 First 30 lines of main.py:

  1: # main.py
  2: 
  3: import os
  4: from contextlib import asynccontextmanager
  5: 
  6: from fastapi import FastAPI, HTTPException, File, UploadFile
  7: from fastapi.middleware.cors import CORSMiddleware
  8: from pydantic import BaseModel
  9: 
 10: from llm.model_loader import ModelLoader
 11: from engines.post_rejection_engine import PostRejectionEngine
 12: from engines.pre_purchase_engine import PrePurchaseEngine
 13: from engines.policy_comparison_engine import PolicyComparisonEngine
 14: 
 15: from schemas.request import PostRejectionRequest, PrePurchaseRequest
 16: from schemas.chat import ReportChatResponse
 17: from schemas.policy_comparison import PolicyComparisonReport
 18: 
 19: from services.report_chat_service import run_report_chat
 20: from services.chat_memory import create_session
 21: from services.document_parser import extract_text_from_file
 22: 
 23: # ── OCR extractor (

---
## 🚀 CELL 8: Start FastAPI Server

**This starts the server in the background.**

Wait for "Server is ready" message before proceeding.

In [30]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/kaggle/working/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache/hub"

print("\U0001f9e0 Pre-downloading MedGemma model...\n")
print("   Model: google/medgemma-4b-it")
print("   Size: ~4-5 GB")
print("   Time: 10-15 minutes (first time only)\n")

# ── HuggingFace Authentication ──
from huggingface_hub import login

hf_token = None

# Method 1: Try Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("\u2705 Got HF_TOKEN from Kaggle Secrets!")
except Exception as e1:
    print(f"   Kaggle Secrets lookup failed: {e1}")

# Method 2: Try environment variable
if not hf_token:
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HF_TOKENS")
    if hf_token:
        print("\u2705 Got HF_TOKEN from environment variable!")

# Method 3: Hardcoded fallback (user can paste their token here)
if not hf_token:
    # \u26a0\ufe0f PASTE YOUR TOKEN BELOW if secrets don't work:
    # hf_token = "hf_YourTokenHere"
    pass

if hf_token:
    login(hf_token)
    print("\u2705 Logged in to HuggingFace!\n")
else:
    print("\u274c HF_TOKEN NOT FOUND!")
    print("   Fix: Go to Kaggle sidebar -> Add-ons -> Secrets")
    print("   Add Label: HF_TOKEN  Value: your huggingface token")
    print("   Then toggle it ON for this notebook and re-run.\n")
    raise RuntimeError("Cannot proceed without HF_TOKEN")

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "google/medgemma-4b-it"

try:
    print("\U0001f4e5 Downloading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("   \u2705 Tokenizer downloaded\n")

    print("\U0001f4e5 Downloading model (this is the slow part)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    print("   \u2705 Model downloaded and cached!\n")

    # Test inference
    print("\U0001f9ea Testing model inference...")
    test_input = "What is insurance waiting period?"
    inputs = tokenizer(test_input, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"   Test query: {test_input}")
    print(f"   Response preview: {response[:100]}...")
    print("\n\u2705 Model is working correctly!\n")

    # Clean up to free memory
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print("\U0001f9f9 Memory cleared for server startup\n")

except Exception as e:
    print(f"\n\u274c Error: {str(e)}\n")
    print("   If this fails, the model will download when server starts.\n")

print("\u2705 Model pre-download complete!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 69.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.4 MB/s eta 0:00:00:00:01
🚀 Starting FastAPI server...

   🧹 Cleaned up old processes

   📝 Server started (PID: 1810)

   ⏳ Waiting for server to be ready...

   (Model loading may take 2-5 minutes)

   [0s] Still loading...

❌ Server process died!

📋 Last 30 lines of log:

    self.tokenizer = AutoTokenizer.from_pretrained(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 628, in from_pretrained
    config = PreTrainedConfig.from_pretrained(pretrained_model_name_or_path, **kwargs)
             ^^^^^^^^^^^^^

---
## 📋 CELL 9: Check Server Logs (If Needed)

In [9]:
# Run this if server didn't start properly
!tail -50 /kaggle/working/server.log


    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1269, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 824, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 424, in main
    run(
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 594, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 67, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/u

---
## 🧪 CELL 10: Test Local Server

In [10]:
import requests
import json

print("🧪 Testing local server...\n")

try:
    # Test root endpoint
    response = requests.get("http://localhost:8000/", timeout=10)
    print("✅ Server is responding!\n")
    print("📊 Response:")
    print(json.dumps(response.json(), indent=2))
    print("\n✅ Local server test passed!\n")
except Exception as e:
    print(f"❌ Server test failed: {str(e)}\n")
    print("   → Check server logs in Cell 9\n")


🧪 Testing local server...

❌ Server test failed: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a9159a64c50>: Failed to establish a new connection: [Errno 111] Connection refused'))

   → Check server logs in Cell 9



---
## 🌐 CELL 11: Setup Ngrok Tunnel

**⚠️ IMPORTANT: Replace `YOUR_NGROK_TOKEN` with your actual token!**

Get token from: https://dashboard.ngrok.com/get-started/your-authtoken

In [11]:
from pyngrok import ngrok
import time

# ⚠️⚠️⚠️ REPLACE THIS WITH YOUR ACTUAL NGROK TOKEN! ⚠️⚠️⚠️
NGROK_AUTH_TOKEN = "3AoGH2qgpya7zUT7YyoxAdZQq0i_AXbPAvDyJJP5A6rbGoVV"

if NGROK_AUTH_TOKEN == "3AoGH2qgpya7zUT7YyoxAdZQq0i_AXbPAvDyJJP5A6rbGoVV":
    print("❌ ERROR: You need to set your ngrok token!\n")
    print("Steps:\n")
    print("1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("2. Sign up (free)")
    print("3. Copy your auth token")
    print("4. Replace 'YOUR_NGROK_TOKEN_HERE' above with your token")
    print("5. Re-run this cell\n")
else:
    print("🌐 Setting up ngrok tunnel...\n")
    
    # Kill any existing tunnels
    ngrok.kill()
    time.sleep(2)
    
    # Set auth token
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    
    # Create tunnel
    try:
        tunnel = ngrok.connect(8000, "http")
        public_url = tunnel.public_url
        
        print("✅ Ngrok tunnel created!\n")
        print("="*60)
        print(f"🌍 PUBLIC URL: {public_url}")
        print("="*60)
        print(f"\n📚 API Documentation: {public_url}/docs")
        print(f"📊 Health Check: {public_url}/")
        print("\n💡 Use this URL in your frontend:\n")
        print(f"   const API_URL = \"{public_url}\"\n")
        print("="*60)
        print("\n⏰ This URL is valid for the current session only")
        print("   (Changes when you restart the notebook)\n")
        
    except Exception as e:
        print(f"❌ Ngrok setup failed: {str(e)}\n")
        print("   → Check if auth token is correct")
        print("   → Verify internet is enabled in Kaggle settings\n")


SyntaxError: invalid decimal literal (47818825.py, line 5)

---
## ✅ CELL 12: Final Health Check

In [ ]:
import requests
import json

if 'public_url' in globals():
    print("🧪 Testing public API...\n")
    
    try:
        response = requests.get(public_url, timeout=10)
        
        if response.status_code == 200:
            print("✅ PUBLIC API IS WORKING!\n")
            print("📊 Server Status:")
            print(json.dumps(response.json(), indent=2))
            print("\n" + "="*60)
            print("🎉 DEPLOYMENT SUCCESSFUL!")
            print("="*60)
            print(f"\n🌍 Your API: {public_url}")
            print(f"📚 Docs: {public_url}/docs")
            print(f"\n💻 Update your frontend with this URL:\n")
            print(f'   const API_URL = "{public_url}"')
            print("\n" + "="*60)
        else:
            print(f"⚠️  API responded with status {response.status_code}")
            
    except Exception as e:
        print(f"❌ Public API test failed: {str(e)}\n")
        print("   → Check if ngrok tunnel is active\n")
else:
    print("⚠️  No public URL found")
    print("   → Run Cell 11 to create ngrok tunnel\n")


---
## 🔄 CELL 13: Restart Server (Without Re-downloading Model)

**Use this if server crashes or you need to restart.**

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/kaggle/working/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache/hub"

print("\U0001f9e0 Pre-downloading MedGemma model...\n")
print("   Model: google/medgemma-4b-it")
print("   Size: ~4-5 GB")
print("   Time: 10-15 minutes (first time only)\n")

# ── HuggingFace Authentication ──
from huggingface_hub import login

hf_token = None

# Method 1: Try Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("\u2705 Got HF_TOKEN from Kaggle Secrets!")
except Exception as e1:
    print(f"   Kaggle Secrets lookup failed: {e1}")

# Method 2: Try environment variable
if not hf_token:
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HF_TOKENS")
    if hf_token:
        print("\u2705 Got HF_TOKEN from environment variable!")

# Method 3: Hardcoded fallback (user can paste their token here)
if not hf_token:
    # \u26a0\ufe0f PASTE YOUR TOKEN BELOW if secrets don't work:
    # hf_token = "hf_YourTokenHere"
    pass

if hf_token:
    login(hf_token)
    print("\u2705 Logged in to HuggingFace!\n")
else:
    print("\u274c HF_TOKEN NOT FOUND!")
    print("   Fix: Go to Kaggle sidebar -> Add-ons -> Secrets")
    print("   Add Label: HF_TOKEN  Value: your huggingface token")
    print("   Then toggle it ON for this notebook and re-run.\n")
    raise RuntimeError("Cannot proceed without HF_TOKEN")

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "google/medgemma-4b-it"

try:
    print("\U0001f4e5 Downloading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("   \u2705 Tokenizer downloaded\n")

    print("\U0001f4e5 Downloading model (this is the slow part)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    print("   \u2705 Model downloaded and cached!\n")

    # Test inference
    print("\U0001f9ea Testing model inference...")
    test_input = "What is insurance waiting period?"
    inputs = tokenizer(test_input, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"   Test query: {test_input}")
    print(f"   Response preview: {response[:100]}...")
    print("\n\u2705 Model is working correctly!\n")

    # Clean up to free memory
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print("\U0001f9f9 Memory cleared for server startup\n")

except Exception as e:
    print(f"\n\u274c Error: {str(e)}\n")
    print("   If this fails, the model will download when server starts.\n")

print("\u2705 Model pre-download complete!")


---
## 📚 Quick Reference Guide

### Common Tasks:

| Task | What to Do |
|------|------------|
| **First time setup** | Run Cells 1-12 in order |
| **Server crashed** | Run Cell 13 (restart), then Cell 11 (new ngrok) |
| **Check errors** | Run Cell 9 (view logs) |
| **Update code from GitHub** | Run Cell 3 (pull), then Cell 13 (restart) |
| **Get new public URL** | Run Cell 11 |
| **Test if working** | Run Cell 12 |

### Important Files:

- **Backend code:** `/kaggle/working/carebridge/`
- **Model cache:** `/kaggle/working/hf_cache/hub/`
- **Server logs:** `/kaggle/working/server.log`

### Troubleshooting:

**Problem:** Server won't start
- Check logs: `!tail -50 /kaggle/working/server.log`
- Verify GPU is enabled in settings
- Try restarting kernel

**Problem:** Ngrok fails
- Verify auth token is correct
- Check internet is enabled
- Try running Cell 11 again

**Problem:** Out of GPU memory
- Restart kernel
- Ensure no other notebooks are running

### Session Limits:

- **GPU quota:** 30 hours/week (free tier)
- **Session timeout:** 9 hours max
- **Auto-shutdown:** 60 minutes of inactivity

### Production Deployment:

For permanent deployment:
- AWS EC2 with GPU (g4dn.xlarge)
- Google Cloud with T4
- Or upgrade ngrok for static domain ($8/month)

---

## 🎉 You're All Set!

Your CareBridge AI backend is now running on Kaggle GPU!

**Next Steps:**
1. Copy the public URL from Cell 11
2. Update your Next.js frontend with the URL
3. Deploy frontend to Vercel
4. Test end-to-end!

**Need help?** Check the troubleshooting guide above or ask your team! 🚀